In [ ]:
# Import Packages
import os
import re
import sys
import time
import copy
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import r2_score, mean_squared_error

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, GroupKFold

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, RidgeCV

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')

In [ ]:
#!chmod 644 ~/.local/share/jupyter/history.sqlite
#!rm ~/.local/share/jupyter/history.sqlite

In [ ]:
EMIT_DATA_CSV         = "../../DATA/AGB_DATA/Merged_Data/EMIT_AGB/AGB_EO_EMIT.csv"
EMIT_MISSING_DATA_CSV = "../../DATA/AGB_DATA/Merged_Data/EMIT_AGB/AGB_EO_EMIT_MISSING.csv"

EMIT_PCA_TRAIN_CSV = "../../DATA/AGB_DATA/Merged_Data/EMIT_AGB/PCA/AGB_EMIT_PCA_TRAIN.csv"
EMIT_PCA_TEST_CSV  = "../../DATA/AGB_DATA/Merged_Data/EMIT_AGB/PCA/AGB_EMIT_PCA_TEST.csv"

SENTINEL_DATA_CSV = "../../DATA/AGB_DATA/Merged_Data/EMIT_AGB/Sentinel_AGB/AGB_EO_SENTINEL.csv"
SENINEL_MISSING_DATA_CSV = "../../DATA/AGB_DATA/Merged_Data/EMIT_AGB/Sentinel_AGB/AGB_VAL_EO_SENTINEL.csv"

# GENERAL FUNCTIONS

In [ ]:
from IPython.display import display

def tabulate_results(results):
    rows = []
    for label, metrics in results.items():
        rows.append({
            'Experiment'         : label,
            'Test R²'            : round(metrics['r2'], 4),
            'RMSE'               : round(metrics['rmse'], 2),
            'CV R² Mean'         : round(metrics['cv_r2_mean'], 4),
            'CV R² Std'          : round(metrics['cv_r2_std'], 4),
            'Group CV R² Mean'   : round(metrics['group_cv_r2_mean'], 4),
            'Group CV R² Std'    : round(metrics['group_cv_r2_std'], 4),
        })

    df = pd.DataFrame(rows)
    df = df.sort_values('Group CV R² Mean', ascending=False).reset_index(drop=True)

    styled = (
        df.style
        .highlight_max(
            subset=['Test R²', 'Group CV R² Mean'],
            color='lightgreen'
        )
        .highlight_min(
            subset=['RMSE', 'Group CV R² Std'],
            color='lightgreen'
        )
    )
    display(styled)

## Residual analysis

In [ ]:
def residual_analysis(y_pred, residuals):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].scatter(y_pred, residuals, alpha=0.4)
    axes[0].axhline(0, color='red', linestyle='--')
    axes[0].set_xlabel('Predicted AGB')
    axes[0].set_ylabel('Residuals')
    axes[0].set_title('Residuals vs Predicted')
    
    axes[1].hist(residuals, bins=30)
    axes[1].set_xlabel('Residual')
    axes[1].set_title('Residual Distribution')
    
    from scipy import stats
    stats.probplot(residuals, dist="norm", plot=axes[2])
    axes[2].set_title('QQ Plot')
    
    plt.tight_layout()
    plt.show()

## CROSS-VALIDATION

### Cross-validation

In [ ]:
def cross_validate(func, X_data, y_data, cv=10, scoring='r2'):
    cv_scores = cross_val_score(func, X_data, y_data, cv=cv, scoring=scoring)
    print("\n Cross-validation ---")
    print(f"CV R² mean : {cv_scores.mean():.4f}")
    print(f"CV R² std  : {cv_scores.std():.4f}")
    print(f"CV scores  : {cv_scores.round(3)}")

    return cv_scores.mean(), cv_scores.std(), cv_scores

### Plot-grouped Cross-validation

In [ ]:
def plot_grouped_cv(func, X_data, y_data, groups, n_splits=10, scoring='r2'):
    gkf    = GroupKFold(n_splits=10)

    cv_scores = cross_val_score(
        func, X_data, y_data,
        cv=gkf.split(X_data, y_data, groups),
        scoring=scoring
    )
    
    print("\nPlot-grouped Cross-validation ---")
    print(f"Plot-grouped CV R² mean : {cv_scores.mean():.4f}")
    print(f"Plot-grouped CV R² std  : {cv_scores.std():.4f}")
    print(f"Plot-grouped CV scores  : {cv_scores.round(3)}")

    return cv_scores.mean(), cv_scores.std(), cv_scores

## Split the data

In [ ]:
def split_data_ver1(X_var, y_var, groups):
    X_local = X_var.copy(deep=True)
    y_local = y_var.copy(deep=True)
    
    from sklearn.model_selection import GroupShuffleSplit
    
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

    train_idx, test_idx = next(gss.split(X_local, y_local, groups=groups))
    
    X_train = X_local.iloc[train_idx]
    X_test  = X_local.iloc[test_idx]
    y_train = y_local.iloc[train_idx]
    y_test  = y_local.iloc[test_idx]

    #print(f"Train rows : {len(X_train)}")
    #print(f"Test rows  : {len(X_test)}")
    
    # Verify no plot appears in both train and test
    train_plots = set(emit_df.iloc[train_idx]['plot_id'])
    test_plots  = set(emit_df.iloc[test_idx]['plot_id'])
    overlap     = train_plots & test_plots
    
    #print(f"Train plots : {len(train_plots)}")
    #print(f"Test plots  : {len(test_plots)}")
    #print(f"Overlapping plots: {len(overlap)}")  # must be 0
    assert not len(overlap)

    return X_train, X_test, y_train, y_test

def split_data_ver2(X_var, y_var):
    X_train, X_test, y_train, y_test = train_test_split(X_var,
                                                        y_var,
                                                        test_size=0.2,
                                                        random_state=42)

    return X_train, X_test, y_train, y_test

## LINEAR REGRESSION FUNCTIONS

In [ ]:
def linear_reg_ver1(X_var, y_var, label):
    groups = X_var['group']
    X_train, X_test, y_train, y_test = split_data_ver1(X_var, y_var, groups)
    
    return run_linear_regression(X_train, X_test, y_train, y_test, label)

def linear_reg_ver2(X_train, X_test, y_train, y_test, label):
    return run_linear_regression(X_train, X_test, y_train, y_test, label)
    
def run_linear_regression(X_train, X_test, y_train, y_test, label):
    X_var = pd.concat([X_train, X_test], axis=0)
    y_var = pd.concat([y_train, y_test], axis=0)

    groups = X_var['group']
    X_var_orig = X_var.copy()
    X_var   = X_var.drop(columns=['group'])
    X_train = X_train.drop(columns=['group'])
    X_test  = X_test.drop(columns=['group'])
    
    scaler  = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    model = LinearRegression()
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)
    r2     = r2_score(y_test, y_pred)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    
    residuals = y_test - y_pred

    print(f"\n--- {label} ---")
    print(f"R²   : {r2:.4f}")
    print(f"RMSE : {rmse:.2f} kg")
    print(f"Features: {X_var.shape[1]}")

    cv_r2_mean, cv_r2_std, cv_scores = cross_validate(LinearRegression(),
                                                      X_var,
                                                      y_var,
                                                      cv=10,
                                                      scoring='r2')

    group_cv_r2_mean, group_cv_r2_std, group_cv_scores = plot_grouped_cv(LinearRegression(),
                                                                         X_var,
                                                                         y_var,
                                                                         groups,
                                                                         n_splits=10,
                                                                         scoring='r2')    

    datum = {"r2": r2,
             "rmse": rmse,
             "y_pred": y_pred,
             "residuals": residuals,
             "cv_r2_mean": cv_r2_mean,
             "cv_r2_std": cv_r2_std,
             "cv_scores": cv_scores,
             "group_cv_r2_mean": group_cv_r2_mean,
             "group_cv_r2_std": group_cv_r2_std,
             "group_cv_scores": group_cv_scores,
             "model": model}

    return datum

## RANDOM FOREST FUNCTIONS

In [ ]:
def randomForest_ver1(X_var, y_var, label):
    groups = X_var['group']
    X_train, X_test, y_train, y_test = split_data_ver1(X_var, y_var, groups)
    return run_randomForest(X_train, X_test, y_train, y_test, label)

def randomForest_ver2(X_train, X_test, y_train, y_test, label):
    return run_randomForest(X_train, X_test, y_train, y_test, label)

def run_randomForest(X_train, X_test, y_train, y_test, label):
    rf = RandomForestRegressor(
        n_estimators    = 500,
        max_features    = 'sqrt',
        min_samples_leaf= 2,
        random_state    = 42,
        n_jobs          = -1
    )

    X_var = pd.concat([X_train, X_test], axis=0)
    y_var = pd.concat([y_train, y_test], axis=0)

    groups  = X_var['group']
    X_var_orig = X_var.copy()
    X_var   = X_var.drop(columns=['group'])
    X_train = X_train.drop(columns=['group'])
    X_test  = X_test.drop(columns=['group'])
    
    # NOTE: Random forest does not need scaling.
    
    rf.fit(X_train, y_train)
    
    # Test set performance
    y_pred = rf.predict(X_test)
    r2     = r2_score(y_test, y_pred)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    residuals = y_test - y_pred

    print(f"EXPERIMENT: {label}")
    print(f"R²   : {r2:.4f}")
    print(f"RMSE : {rmse:.2f} kg")
    print(f"Features: {X.shape[1]}")

    cv_r2_mean, cv_r2_std, cv_scores = cross_validate(rf,
                                                      X_var,
                                                      y_var,
                                                      cv=10,
                                                      scoring='r2')
    
    group_cv_r2_mean, group_cv_r2_std, group_cv_scores = plot_grouped_cv(rf,
                                                                         X_var,
                                                                         y_var,
                                                                         groups,
                                                                         n_splits=10,
                                                                         scoring='r2')
    
    importances = pd.Series(rf.feature_importances_, index=X_var.columns)

    datum = {"r2": r2,
             "rmse": rmse,
             "y_pred": y_pred,
             "residuals": residuals,
             "cv_r2_mean": cv_r2_mean,
             "cv_r2_std": cv_r2_std,
             "cv_scores": cv_scores,
             "group_cv_r2_mean": group_cv_r2_mean,
             "group_cv_r2_std": group_cv_r2_std,
             "group_cv_scores": group_cv_scores,
             "importances": importances,
             "model": rf}

    return datum

In [ ]:
def show_importances(results):
    importances = results["importances"].sort_values(ascending=False)
    
    # Feature importances
    N = 4
    print(f"\nTop {N} feature importances:")
    for feat, imp in importances.head(N).items():
        bar = '█' * int(imp * 50)
        print(f"  {feat:45s} {imp:.4f}  {bar}")

# EMIT FULL BANDS DATA

In [ ]:
emit_df = pd.read_csv(EMIT_DATA_CSV)
print(emit_df.shape)

## DATA PREPROCESSING

### Select feature columns

In [ ]:
non_feature_cols = [
    'plant_AGB_kg',        # Target variable
    'dataset',             # metadata
    'EMIT_selected_date',  # metadata
    'EMIT_granule',        # metadata
    'start_date',          # metadata
    'end_date',            # metadata
    'capture_start',       # metadata
    'capture_end',         # metadata
    'latitude',            # coordinate
    'longitude',           # coordinate
]
target = 'plant_AGB_kg'

feature_cols = [c for c in emit_df.columns if c not in non_feature_cols]

X = emit_df[feature_cols]
y = emit_df[target]

print(f"Features : {len(feature_cols)}")
print(f"Rows     : {len(emit_df)}")

### Handle categorical variables.

In [ ]:
# Preserve original plot_id for grouping.
X = X.copy()
X['group'] = X['plot_id']

In [ ]:
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
categorical_cols = [c for c in categorical_cols if c != 'group']
print(f"Categorical columns: {categorical_cols}")

In [ ]:
#Convert categorical variables into one-hot encoding.
X = pd.get_dummies(X, columns=categorical_cols, dtype=int)

### Handle NULL data

In [ ]:
print(X.isnull().sum().sum())

In [ ]:
null_rows = X[X.isnull().any(axis=1)]

print(f"Rows with at least one NULL: {len(null_rows)}")
print(f"Total rows                 : {len(X)}")
print(f"Percentage                 : {len(null_rows)/len(X)*100:.1f}%")

In [ ]:
# NULL count per column for only the affected rows
null_col_counts = null_rows.isnull().sum().sort_values(ascending=False)

print("\nNULL count per column in affected rows:")
print(null_col_counts[null_col_counts > 0])

These 12 bands are all in the 1300–1800 nm range — the shortwave infrared (SWIR) region. EMIT masks out specific wavelength ranges that are dominated by atmospheric water vapor absorption, where the signal is unreliable. The two main atmospheric water vapor absorption windows in EMIT are:  

~1340 to 1460 nm  ← water vapor absorption band  
~1780 to 1970 nm  ← water vapor absorption band  

All of the 12 null columns listed above, fall squarely in these two ranges. EMIT sets these to nodata for pixels where atmospheric correction failed or where the absorption is too strong to recover reliable surface reflectance.  

The above bands are unreliable by design. Drop them.

In [ ]:
vapor_bands = null_rows.isnull().sum()
vapor_bands = vapor_bands[vapor_bands > 0].index.tolist()

print(f"Dropping {len(vapor_bands)} water vapor absorption bands:")
print(vapor_bands)

X = X.drop(columns=vapor_bands)
print(f"\nNULL count after dropping: {X.isnull().sum().sum()}")

### Remove Low Variance Features (cols)

In [ ]:
from sklearn.feature_selection import VarianceThreshold

encoded_cols    = [c for c in X.columns if c.startswith('plot_id_') or c.startswith('species_') or c.startswith('group')]
continuous_cols = [c for c in X.columns if c not in encoded_cols]

# Apply variance threshold only to continuous columns
X_continuous = X[continuous_cols]
selector     = VarianceThreshold(threshold=0.01)
selector.fit(X_continuous)

low_variance_cols = X_continuous.columns[~selector.get_support()].tolist()
print(f"Low variance columns removed: {len(low_variance_cols)}")

kept_continuous = X_continuous.columns[selector.get_support()].tolist()

# Reassemble — keep all encoded columns, keep only high-variance continuous columns
X = X[kept_continuous + encoded_cols]
print(f"Features after variance filtering: {len(X.columns)}")

### Remove Features With Weak Correlation to Target

In [ ]:
# Separate continuous and encoded columns
encoded_cols    = [c for c in X.columns if c.startswith('plot_id_') or c.startswith('species_') or c.startswith('group')]
continuous_cols = [c for c in X.columns if c not in encoded_cols]

# Apply correlation filter only to continuous columns
X_continuous = X[continuous_cols]
correlations = X_continuous.corrwith(y).abs().sort_values(ascending=False)

# Compute Pearson correlation of every feature against target
CORR_THRESHOLD   = 0.1
strong_corr_cols = correlations[correlations >= CORR_THRESHOLD].index.tolist()
weak_corr_cols   = correlations[correlations <  CORR_THRESHOLD].index.tolist()

print(f"Continuous features kept   : {len(strong_corr_cols)}")
print(f"Continuous features removed: {len(weak_corr_cols)}")
print(f"\nTop 20 correlated features:")
print(correlations.head(20))

# Reassemble — keep all encoded columns, keep only high-correlation continuous columns
X = X[strong_corr_cols + encoded_cols]
print(f"\nTotal features after correlation filtering: {len(X.columns)}")

In [ ]:
[c for c in X.columns if c.startswith('group')]

In [ ]:
assert(len(X.filter(regex=r'^species').columns.tolist()))
assert(len(X.filter(regex=r'^plot').columns.tolist()))

# LINEAR REGRESSION

In [ ]:
linear_reg_experiments = {}

## Linear regression model with structural variables only.

### Linear regression modeling with all of the structural variables excluding plot_id variable.

In [ ]:
label_1 = "Structural variables without plot_id"

emit_cols  = [c for c in X.columns if c.startswith('EMIT')]
plot_cols  = [c for c in X.columns if c.startswith('plot_id')]
other_cols = [c for c in X.columns if c not in emit_cols and c not in plot_cols]
X_struct_wo_plot = X[other_cols]

In [ ]:
results = linear_reg_ver1(X_struct_wo_plot, y, label_1)
results["X_data"] = X_struct_wo_plot.copy()

# Cross-validation to confirm that the R^2 above is not by coincidence.
linear_reg_experiments["LIN: " + label_1] = results

### Linear regression modeling with all of the structural variables including plot_id variable.

In [ ]:
label_2 = "Structural variables + plot_id"

emit_cols  = [c for c in X.columns if c.startswith('EMIT')]
other_cols = [c for c in X.columns if c not in emit_cols]

X_struct_wi_plot = X[other_cols]

results = linear_reg_ver1(X_struct_wi_plot, y, label_2)
results["X_data"] = X_struct_wi_plot.copy()

# Cross-validation to confirm that the R^2 above is not by coincidence.
linear_reg_experiments["LIN: " + label_2] = results

### Linear regression modeling with structural variables and interaction terms.

In [ ]:
X_int = X_struct_wo_plot.copy(deep=True)
species_cols = [c for c in X.columns if c.startswith('species_')]

# Create interaction terms
for species_col in species_cols:
    X_int[f'diameter_x_{species_col}'] = X_int['diameter'] * X_int[species_col]
    X_int[f'height_x_{species_col}']   = X_int['height']   * X_int[species_col]

# diameter x height
X_int['diameter_x_height'] = X_int['diameter'] * X_int['height']

print(f"Features after adding interaction terms: {len(X_int.columns)}")
print(f"Interaction terms added: {len(species_cols) * 2 + 1}")

In [ ]:
label_3 = "Structural variables + interaction terms"

results = linear_reg_ver1(X_int, y, label_3)
results["X_data"] = X_int.copy()

# Cross-validation to confirm that the R^2 above is not by coincidence.
linear_reg_experiments["LIN: " + label_3] = results

## Linear regression modeling with EMIT bands only

In [ ]:
label_4 = "EMIT bands only"

emit_cols  = [c for c in X.columns if c.startswith('EMIT')]
X_emit_bands = X[emit_cols + ["group"]]

results = linear_reg_ver1(X_emit_bands, y, label_4)
results["X_data"] = X_emit_bands.copy()

# Cross-validation to confirm that the R^2 above is not by coincidence.
linear_reg_experiments["LIN: " + label_4] = results

## Linear regression modeling with all of the variables

### Linear regression modeling with all of the variables excluding "plot_id" variable.

In [ ]:
label_5 = "All variables (structural + EMIT) without plot_id"

plot_cols  = [c for c in X.columns if c.startswith('plot_id_')]
other_cols = [c for c in X.columns if c not in plot_cols]

X_full_without_plot = X[other_cols]

results = linear_reg_ver1(X_full_without_plot, y, label_5)
results["X_data"] = X_full_without_plot.copy()

# Cross-validation to confirm that the R^2 above is not by coincidence.
linear_reg_experiments["LIN: " + label_5] = results

### Linear regression modeling with all of the variables including "plot_id" variable.

In [ ]:
label_6 = "All variables (structural + EMIT) + plot_id"
results = linear_reg_ver1(X, y, label_6)
results["X_data"] = X.copy()

# Cross-validation to confirm that the R^2 above is not by coincidence.
linear_reg_experiments["LIN: " + label_6] = results

### Linear regression modeling with all of the variables WITH interaction terms

In [ ]:
X_int_2 = X.copy(deep=True)

plot_cols  = [c for c in X_int_2.columns if c.startswith('plot_id_')]
X_int_2 = X_int_2.drop(columns=plot_cols)

species_cols = [c for c in X_int_2.columns if c.startswith('species_')]

# Create interaction terms
for species_col in species_cols:
    X_int_2[f'diameter_x_{species_col}'] = X_int_2['diameter'] * X_int_2[species_col]
    X_int_2[f'height_x_{species_col}']   = X_int_2['height']   * X_int_2[species_col]

# diameter x height
X_int_2['diameter_x_height'] = X_int_2['diameter'] * X_int_2['height']

print(f"Features after adding interaction terms: {len(X_int.columns)}")
print(f"Interaction terms added: {len(species_cols) * 2 + 1}")

In [ ]:
label_7 = "All variables (structural + EMIT) + interaction terms"

results = linear_reg_ver1(X_int_2, y, label_7)
results["X_data"] = X_int_2.copy()

# Cross-validation to confirm that the R^2 above is not by coincidence.
linear_reg_experiments["LIN: " + label_7] = results

#### Check statistical significance of EMIT bands contribution

In [ ]:
# Run both models with k-fold cross validation
cv_r2_selected = cross_val_score(
    LinearRegression(), X_int_2.drop(columns=['group']), y, cv=10, scoring='r2'
)

cv_r2_full = cross_val_score(
    LinearRegression(), X.drop(columns=['group']), y, cv=10, scoring='r2'
)

print(f"Structural R² : {cv_r2_selected.mean():.4f} ± {cv_r2_selected.std():.4f}")
print(f"Full R²       : {cv_r2_full.mean():.4f} ± {cv_r2_full.std():.4f}")

# Paired t-test — tests whether the difference in R² across folds is significant
t_stat, p_value = stats.ttest_rel(cv_r2_full, cv_r2_selected)

print(f"\nPaired t-test:")
print(f"t-statistic : {t_stat:.4f}")
print(f"p-value     : {p_value:.4f}")

if p_value < 0.05:
    print("The improvement from adding EMIT bands is statistically significant")
else:
    print("The improvement from adding EMIT bands is NOT statistically significant")

## PICK THE BEST LINEAR REGRESSION MODEL
Select the model with the highest cv_r2_mean.

### Summary of experiments so far.

In [ ]:
tabulate_results(linear_reg_experiments)

In [ ]:
max_val = max(v["cv_r2_mean"] for v in linear_reg_experiments.values())

lin_best_labels = [
    label
    for label, vals in linear_reg_experiments.items()
    if vals["cv_r2_mean"] == max_val
]

print(lin_best_labels)

**NOTE:** I am selecting "'All variables (structural + EMIT) + interaction terms'" assuming that EMIT bands might add some extra explanation.

In [ ]:
#best_label = 'All variables (structural + EMIT) + interaction terms'
lin_best_label = lin_best_labels[0]
lin_X_selected = linear_reg_experiments[lin_best_label]["X_data"]

print(f"Best model: {lin_best_label}")
print(f"Test R2    : {linear_reg_experiments[lin_best_label]["r2"]}")
print(f"CV R2 mean : {linear_reg_experiments[lin_best_label]["cv_r2_mean"]}")
print(f"CV Scores  : {linear_reg_experiments[lin_best_label]["cv_scores"]}")

### ANALYSIS OF THE BEST LINEAR REGRESSION MODEL

**R2 value**
 - The test R² (0.63) is lower than the CV mean. 
 - It means the 12 test plots happen to be harder to predict than average.
 - This is not a bug, just the natural variation from holding out a specific subset of plots.

**Grouped Cross-validation**
 - The CV mean of 0.78 across all 10 fold combinations is the more reliable estimate of true generalization performance.
 - Some folds perform well (0.898, 0.842, 0.819) and some like Fold 7 (0.158) and Fold 3 (0.657) are significantly weaker than the rest.  
 - The other 8 folds cluster between 0.81 and 0.95.  
 - This pulls the mean down to 0.78 and inflates the standard deviation of R^2 to 0.22.  
 - Those two weak folds almost certainly contain plots with species or diameter ranges underrepresented in training.

### Residual analysis

In [ ]:
y_pred    = linear_reg_experiments[lin_best_label]["y_pred"]
residuals = linear_reg_experiments[lin_best_label]["residuals"]

residual_analysis(y_pred, residuals)

### Check if regularization adds any value.

In [ ]:
groups = lin_X_selected['group']
lin_X_selected = lin_X_selected.drop(columns=['group'])

X_train, X_test, y_train, y_test = split_data_ver1(lin_X_selected, y, groups)
                                                    
# RidgeCV automatically finds the best regularization strength
ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge',  RidgeCV(alphas=[0.01, 0.1, 1, 10, 100, 1000]))
    #('ridge',  RidgeCV(alphas=[1, 10, 100, 1000, 10000, 100000]))
])

gkf = GroupKFold(n_splits=10)

# Cross validation
cv_scores = cross_val_score(
    ridge_pipeline,
    lin_X_selected,
    y,
    #cv=10,
    cv=gkf.split(lin_X_selected, y, groups),
    scoring='r2'
)

print(f"Ridge CV R² mean : {cv_scores.mean():.4f}")
print(f"Ridge CV R² std  : {cv_scores.std():.4f}")
print(f"Ridge CV scores  : {cv_scores.round(3)}")

# Fit on train set to inspect best alpha and test performance
ridge_pipeline.fit(X_train, y_train)
y_pred = ridge_pipeline.predict(X_test)

r2   = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"\nTest R²   : {r2:.4f}")
print(f"Test RMSE : {rmse:.2f} kg")
print(f"Best alpha: {ridge_pipeline.named_steps['ridge'].alpha_}")

**Ridge CV**
 - The Ridge CV scores are now identical to the plain linear regression plot-grouped CV scores.  
 - The best alpha of 0.01 means Ridge chose virtually no regularization.  
 - Both models are identical in practice.

# RANDOM FOREST.

In [ ]:
random_forest_experiments={}

## RandomForest model with just structural variables excluding plot_id variable.

In [ ]:
label_1 = "Structural variables without plot_id"
results = randomForest_ver1(linear_reg_experiments["LIN: " + label_1]["X_data"], y, label_1)
results["X_data"] = linear_reg_experiments["LIN: " + label_1]["X_data"].copy()

random_forest_experiments["RF: " +label_1] = results

show_importances(results)

## RandomForest model with just structural variables including plot_id variable.

In [ ]:
label_2 = "Structural variables + plot_id"
results = randomForest_ver1(linear_reg_experiments["LIN: " + label_2]["X_data"], y, label_2)
results["X_data"] = linear_reg_experiments["LIN: " + label_2]["X_data"].copy()

random_forest_experiments["RF: " +label_2] = results

show_importances(results)

## RandomForest model with just structural variables along with interaction terms.

In [ ]:
label_3 = 'Structural variables + interaction terms'
results = randomForest_ver1(linear_reg_experiments["LIN: " + label_3]["X_data"], y, label_3)
results["X_data"] = linear_reg_experiments["LIN: " + label_3]["X_data"].copy()
random_forest_experiments["RF: " +label_3] = results

show_importances(results)

## RandomForest model with just EMIT Bands.

In [ ]:
label_4 = "EMIT bands only"
results = randomForest_ver1(linear_reg_experiments["LIN: " + label_4]["X_data"], y, label_4)
results["X_data"] = linear_reg_experiments["LIN: " + label_4]["X_data"].copy()
random_forest_experiments["RF: " +label_4] = results

show_importances(results)

## RandomForest model with all of the variables excluding plot_id variable.

In [ ]:
label_5 = "All variables (structural + EMIT) without plot_id"
results = randomForest_ver1(linear_reg_experiments["LIN: " + label_5]["X_data"], y, label_5)
results["X_data"] = linear_reg_experiments["LIN: " + label_5]["X_data"].copy()
random_forest_experiments["RF: " +label_5] = results

show_importances(results)

## RandomForest model with all of the variables including plot_id variable.

In [ ]:
label_6 = "All variables (structural + EMIT) + plot_id"
results = randomForest_ver1(linear_reg_experiments["LIN: " + label_6]["X_data"], y, label_6)
results["X_data"] = linear_reg_experiments["LIN: " + label_6]["X_data"].copy()
random_forest_experiments["RF: " +label_6] = results

show_importances(results)

## RandomForest model with all of the variables along with interaction terms.

In [ ]:
label_7 = "All variables (structural + EMIT) + interaction terms"
results = randomForest_ver1(linear_reg_experiments["LIN: " + label_7]["X_data"], y, label_7)
results["X_data"] = linear_reg_experiments["LIN: " + label_7]["X_data"].copy()
random_forest_experiments["RF: " +label_7] = results

show_importances(results)

## Pick the best RandomForest model.
Pick the model with the highest Cross-Validation R2 mean value.

### Summary of experiments so far.

In [ ]:
tabulate_results(random_forest_experiments)

**SELECTED RANDOM FOREST MODEL**

In [ ]:
max_val = max(v["cv_r2_mean"] for v in random_forest_experiments.values())

rf_best_labels = [
    label
    for label, vals in random_forest_experiments.items()
    if vals["cv_r2_mean"] == max_val
]

print(rf_best_labels)
print("max cv_r2_mean =", max_val)

In [ ]:
rf_best_label = rf_best_labels[0]

rf_X_selected = random_forest_experiments[rf_best_label]["X_data"]

print("Random Forest best model")
print(f"Test R2    : {random_forest_experiments[rf_best_label]["r2"]}")
print(f"CV R2 mean : {random_forest_experiments[rf_best_label]["cv_r2_mean"]}")
print(f"CV Scores  : {random_forest_experiments[rf_best_label]["cv_scores"]}")

In [ ]:
show_importances(random_forest_experiments[rf_best_label])

## Analysis

**R2 vs CV R2 mean**  
| Data type | Group CV | std |
| :--- | :----: | ---: |
|All variables (structural + EMIT) without plot_id | 0.4538 |   0.5260|
|Structural + interactions | 0.7782 | 0.2244|

All + EMIT is 0.33 points worse in generalization and 2.3 times more variable. This means:
 - On average it explains 45% of variance on unseen plots versus 78% for structural + interactions
 - Its performance swings wildly depending on which plots are held out
 - Some folds produce negative R² (-0.775, -0.220) meaning it actively fails on certain plot types
 - The tiny 0.006 test R² advantage for All + EMIT is not worth the massive generalization penalty.

**Feature importance**  
 - Diameter alone explains 38.59% of AGB variance  
 - Interaction term "diameter_x_species_Avicennia germinans" explains 20.64% of AGB variance
 - Interaction term "diameter_x_height" explains 18.63% of AGB variance

**NOTE:** The importance of **height** has dropped. But this is not because **height** is insignificant. It is because of the *importance splitting* with the interaction term **diameter_x_species_Avicennia germinans**.

The above two variables together account for 77.5% of the model's explanatory power.  

### Residual analysis

In [ ]:
y_pred    = random_forest_experiments[rf_best_label]["y_pred"]
residuals = random_forest_experiments[rf_best_label]["residuals"]

residual_analysis(y_pred, residuals)

# EMIT PCA DATA

In [ ]:
emit_pca_train_df = pd.read_csv(EMIT_PCA_TRAIN_CSV)
emit_pca_test_df  = pd.read_csv(EMIT_PCA_TEST_CSV)

In [ ]:
non_feature_cols = [
    'plant_AGB_kg',        # Target variable
    #'plot_id',             # metadata
    'dataset',             # metadata
    'EMIT_selected_date',  # metadata
    'EMIT_granule',        # metadata
    'start_date',          # metadata
    'end_date',            # metadata
    'capture_start',       # metadata
    'capture_end',         # metadata
    'latitude',            # coordinate
    'longitude',           # coordinate
]

# Drop raw EMIT bands — keep only PCA components
emit_raw_cols = [c for c in emit_pca_train_df.columns if re.match(r'^EMIT_R\d+', c)]

# Training features and target
X_train = emit_pca_train_df.drop(columns=non_feature_cols + emit_raw_cols)
y_trian = emit_pca_train_df['plant_AGB_kg']

# Test features and target
X_test  = emit_pca_test_df.drop(columns=non_feature_cols + emit_raw_cols)
y_test  = emit_pca_test_df['plant_AGB_kg']

print(f"Features : {len(X_train.columns)}")
print(f"Columns  : {list(X_train.columns)}")

In [ ]:
# One-hot encode species
X_train = pd.get_dummies(X_train, columns=['species'], dtype=int)
X_test  = pd.get_dummies(X_test,  columns=['species'], dtype=int)

# Align columns — test may have different species columns
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

In [ ]:
# Preserve original plot_id for grouping.
X_train = X_train.rename(columns={'plot_id': 'group'})
X_test = X_test.rename(columns={'plot_id': 'group'})
# REVISIT THIS.

In [ ]:
# Add interaction terms
species_cols = [c for c in X_train.columns if c.startswith('species_')]

for col in species_cols:
    X_train[f'diameter_x_{col}'] = X_train['diameter'] * X_train[col]
    X_test[f'diameter_x_{col}']  = X_test['diameter']  * X_test[col]
    X_train[f'height_x_{col}']   = X_train['height']   * X_train[col]
    X_test[f'height_x_{col}']    = X_test['height']    * X_test[col]

X_train['diameter_x_height'] = X_train['diameter'] * X_train['height']
X_test['diameter_x_height']  = X_test['diameter']  * X_test['height']

In [ ]:
pca_results = {}

## LINEAR REGRESSION ON PCA

In [ ]:
label = "Structural variables + Interaction terms + PCA components"
result = linear_reg_ver2(X_train, X_test, y_train, y_test, label)
pca_results["PCA: " + label] = result

### ANALYSIS

In [ ]:
print(f"Previous best Linear regression: {lin_best_label}")
print(f"Test R2         : {linear_reg_experiments[lin_best_label]["r2"]}")
print(f"CV R2 mean      : {linear_reg_experiments[lin_best_label]["cv_r2_mean"]}")
print(f"CV R2 std       : {linear_reg_experiments[lin_best_label]["cv_r2_std"]}")
print(f"Group CV R2 mean: {linear_reg_experiments[lin_best_label]["group_cv_r2_mean"]}")
print(f"Group CV R2 std : {linear_reg_experiments[lin_best_label]["group_cv_r2_std"]}")

**Compare with previous best:**
| Data type | Test R^2 | Group CV Mean | Group CV std |
| :--- | :----: | ---: | ---: |
|Structural + interactions| 0.6339| 0.7781| 0.2244|
|Structural + interactions + PCA| 0.5094| 0.7717| 0.2420|

**REVISIT**

**Group CV improved from 0.7782 to 0.8065**  
Adding 4 PCA components to the structural + interaction model improved honest generalization by 0.028 points.  
This is modest but consistent — the PCA components are adding a small amount of genuine signal beyond structural variables alone.

**Group CV std improved from 0.2244 to 0.1756**  
The model is more stable across fold combinations.  
This is actually more meaningful than the mean improvement — it means the PCA components help the model generalize more consistently across different plot types, not just on average.

**Test R² dropped from 0.6339 to 0.5094**  
The 12 held-out test plots happen to be harder to predict with PCA components included.  
This is a single split artifact — the group CV is the reliable number here.


**Minimum Fold Score (Minimum of CV scores)**  
Previous best minimum fold : 0.158  
PCA model minimum fold     : 0.418   
 - CV scores: [0.905, 0.642, 0.943, 0.595, 0.896, 0.937, 0.894, 0.902, 0.932, 0.418]

The worst-case fold improved substantially.  
The PCA components are helping the model handle the previously problematic plot combinations — the ones that collapsed to 0.158 in the structural-only model.

## RANDOM FOREST ON PCA

In [ ]:
label_rf = "All variables (structural + EMIT) + PCA"
result = randomForest_ver2(X_train, X_test, y_train, y_test, label_rf)
show_importances(results)

pca_results["PCA: " + label_rf] = result

In [ ]:
tabulate_results(pca_results)

### Analysis

**Compare with previous best**

In [ ]:
print(f"Previous best Random Forest model: {rf_best_label}")
print(f"Test R2    : {random_forest_experiments[rf_best_label]["r2"]}")
print(f"CV R2 mean : {random_forest_experiments[rf_best_label]["cv_r2_mean"]}")
print(f"CV R2 mean : {random_forest_experiments[rf_best_label]["cv_r2_std"]}")

| Data type | Test R^2 | CV Mean | CV Std| Group CV Mean | Group CV std |
| :--- | :----: | ---: | ---: | ---: | ---: |
|RF: Structural + interactions| 0.9145| 0.8937| 0.1004| 0.9532| 0.0429|
|RF: All + EMIT + PCA| 0.8991| 0.8644| 0.0908| 0.9286| 0.0507|

Structural + interactions is the better model on every metric.
 - Test R² is higher (0.9145 vs 0.8991)  
 - CV mean is higher (0.8937 vs 0.8644)  
 - Group CV mean is higher (0.9532 vs 0.9286) <- Important  
 - Group Std is lower (0.0429 vs 0.0507)

It looks like adding EMIT bands data in any form (direct data or PCA components) to the structural variables is hurting the performance.

**Let us look at the Feature Importances.**  
**Random Forest (Data = Structural variables + Interaction terms):**  
| Feature | Importance |
| :--- | :----: |
  |diameter|                  0.3859 | 
  |diameter_x_Avicennia|      0.2064 |
  |diameter_x_height|         0.1863 | 
  |height|                    0.0803 | 

**Random Forest (Data = Structural variables + EMIT + PCA components):**  
| Feature | Importance |
| :--- | :----: |
  |diameter |                 0.2425  |
  |diameter_x_Avicennia|      0.1877  |
  |diameter_x_height|         0.1822  |
  |height|                    0.0930  |

It appears that adding PCA components is diluting the importance of structural variables — diameter drops from 0.39 to 0.24.  
The model is spreading its attention across more features without gaining predictive power.  
The PCA components are absorbing importance that was previously concentrated in the dominant structural signal.

# FINAL ANALYSIS

In [ ]:
total_experiments = {**linear_reg_experiments, **random_forest_experiments, **pca_results}
tabulate_results(total_experiments)

**NOTE:**  
Andre's global model achieved R²=0.36 with 524 grid cells and 5 climate variables.  

**Structural only — R² = 0.84, 6 features**  
Diameter, height, species, wood density alone explain 84% of variance.  

**EMIT bands only — R² = 0.25, 157 features**  
 - Spectral data alone is a weak predictor of individual tree AGB.  
 - This makes sense — a single 60m EMIT pixel captures the canopy of many trees mixed together, not one individual tree.
 - The spectral signal is a stand-level measurement being used to predict a tree-level target.  
 - The resolution mismatch is the fundamental problem here.

**Structural + EMIT — R² = 0.87, 163 features**  
 - Adding 157 EMIT bands to 6 structural variables improves R² by only 0.027.
 - The marginal contribution of EMIT is tiny relative to its feature count.

**plot_id paradox**  
 - Adding plot_id improves EMIT-only performance but hurts or does nothing when structural variables are present.
 - This confirms structural variables already capture what plot_id was proxying for.

**FINALLY**  
 - The above confirms that  diameter, height, and species are the dominant drivers of individual tree AGB.
 - Remote sensing appears to adds marginal value at the individual tree level.

**https://onlinelibrary.wiley.com/doi/10.1002/ldr.70594?af=R**
A study of 109 mangrove trees across eight species in Timor-Leste found very strong positive correlations between DBH and biomass variables (r = 0.963–0.971), indicating that tree diameter represents a reliable predictor of biomass and carbon storage. 

**https://pmc.ncbi.nlm.nih.gov/articles/PMC12425185/**
A study using 302 destructively sampled mangrove trees found that the single parameter with the strongest predictive ability in most studies is diameter at breast height (DBH), and that DBH explains a lot of variability in other tree metrics such as height or aboveground biomass. 

**https://www.sciencedirect.com/science/article/abs/pii/S0304377007001829**
Komiyama et al. (2008) in their widely cited review of mangrove allometry state that trunk diameter of a tree is highly correlated with trunk weight, and that since tree diameter is easy to measure but tree weight is much more difficult to determine, this gives a relatively easy way to estimate biomass. 

**https://www.sciencedirect.com/science/article/abs/pii/S0272771420307022**
Chave et al. (2005) reported that allometric equations with total tree height yielded less biased estimates of AGB, though tree height has often been ignored in carbon-accounting programs because measuring tree height accurately is difficult in mangrove forests. 